# Matrix Exponentials: Time Evolution and Quantum Dynamics

The matrix exponential e^A is the **bridge between quantum mechanics differential equations and quantum gates**. It describes how quantum states evolve under Hamiltonians.

## Definition: Power Series Expansion

For a square matrix A:
$$e^A = I + A + \frac{A^2}{2!} + \frac{A^3}{3!} + \ldots = \sum_{n=0}^\infty \frac{A^n}{n!}$$

This series **always converges** for matrices, regardless of A.

**Key property**: e^A is always invertible with (e^A)⁻¹ = e^{-A}.

## The Schrödinger Equation: Where Matrix Exponentials Arise

In quantum mechanics, state evolution follows:
$$i\hbar \frac{d}{dt}|\psi(t)\rangle = H|\psi(t)\rangle$$

where H is the **Hamiltonian** (total energy operator).

**Solution**: 
$$|\psi(t)\rangle = e^{-iHt/\hbar}|\psi(0)\rangle = U(t)|\psi(0)\rangle$$

The **time evolution operator** U(t) = e^{-iHt/ℏ} is unitary by construction:
$$U(t)^\dagger U(t) = e^{iHt/\hbar} e^{-iHt/\hbar} = I$$

(since H is Hermitian, so (e^{-iHt})† = e^{iHt}).

This is the ONLY way quantum states evolve! (Between measurements, which are instantaneous.)

## Computing Matrix Exponentials: Spectral Method

For Hermitian H with eigendecomposition H = P Λ P†:
$$e^{-iHt/\hbar} = P \cdot e^{-i\Lambda t/\hbar} \cdot P^\dagger = P \begin{pmatrix} e^{-i\lambda_1 t/\hbar} & 0 \\ 0 & e^{-i\lambda_2 t/\hbar} \end{pmatrix} P^\dagger$$

The exponential of the diagonal matrix is just exponentials on the diagonal!

**Example**: For σz = |0⟩⟨0| - |1⟩⟨1|:
$$e^{-i\sigma_z t} = e^{-it}|0\rangle\langle 0| + e^{it}|1\rangle\langle 1| = \begin{pmatrix} e^{-it} & 0 \\ 0 & e^{it} \end{pmatrix}$$

## Quantum Gate Generation: H = (Pauli Matrices)

Every single-qubit gate can be expressed as:
$$e^{-i\theta \vec{n} \cdot \vec{\sigma}} = I \cos(\theta) - i(\vec{n} \cdot \vec{\sigma})\sin(\theta)$$

where σ = (σₓ, σᵧ, σz) and n is a unit vector.

**Examples**:
- **RX(θ)** = e^{-iθσₓ/2} (rotation about x-axis)
- **RZ(θ)** = e^{-iθσz/2} (rotation about z-axis)
- **RY(θ)** = e^{-iθσᵧ/2} (rotation about y-axis)

These rotations are **exactly** time evolution under σ gates!

## Trotter-Suzuki Decomposition: Approximating Complex Evolution

For Hamiltonians with multiple terms H = A + B, exact evolution e^{-i(A+B)t} is hard. But:
$$e^{-i(A+B)t} \approx [e^{-iAt/n} e^{-iBt/n}]^n$$

splits complex evolution into simpler pieces—applying e^{-iAt} then e^{-iBt} repeatedly.

This is **fundamental** for variational quantum algorithms (VQE, QAOA):
1. Propose ansatz: e^{-iθ₁σz} e^{-iθ₂σₓ} ... e^{-iθₘσz}
2. Find optimal angles {θᵢ}
3. Run on quantum hardware

The ansatz is a **product of matrix exponentials of Pauli terms**!

## Dynamical Decoupling: Canceling Unwanted Evolution

Sometimes you want to cancel evolution under unwanted Hamiltonian. If:
$$H_{\text{unwanted}} = H_z$$

You can apply refocusing pulses: e^{-iπσₓ/2} to flip the evolution, canceling it.

More generally, **dynamical decoupling** sequences like UHRIG use pulses to zero out noise operators.

## Exponential of Commuting Matrices

If [A,B] = 0 (commute), then:
$$e^{A+B} = e^A e^B$$

This simplification is WHY commuting observables are special in quantum mechanics!

For non-commuting A,B:
$$e^{A+B} \approx e^A e^B e^{-\frac{1}{2}[A,B]} \quad \text{(Baker-Campbell-Hausdorff formula)}$$

This gives higher-order correction terms involving commutators.

## Matrix Functions Beyond Exponentials

Once you understand e^A, you can define:
- **sin(A)** = (e^{iA} - e^{-iA})/(2i)
- **cos(A)** = (e^{iA} + e^{-iA})/2
- **log(A)** = ∫ₐ^∞ (λI - A)⁻¹ dλ

All obey standard calculus rules!

## Numerical Stability: When Computation Gets Hard

For large times t and high-dimensional systems:
- e^{-iHt} requires many Trotter steps
- Eigenvalue computation of H becomes bottleneck
- For n qubits, H is 2ⁿ × 2ⁿ (exponential size!)

This is why **variational quantum algorithms** replace precise time evolution with parametric circuits optimized classically.

In [ ]:
# Functional code: Matrix exponentials and time evolution
import numpy as np
from scipy.linalg import expm

# Define Pauli matrices
sigma_x = np.array([[0, 1], [1, 0]])
sigma_z = np.array([[1, 0], [0, -1]])

# Time evolution with Hamiltonian H = σz
t = 0.5
H = sigma_z
U_t = expm(-1j * H * t)  # e^{-iHt}

# Spectral method: compute e^{-iΛt} in eigenbasis
evals, evecs = np.linalg.eigh(H)
exp_diag = np.diag(np.exp(-1j * evals * t))
U_t_spectral = evecs @ exp_diag @ evecs.conj().T

# RZ(θ) gate: e^{-iθσz/2}
theta = np.pi / 4
RZ = expm(-1j * theta * sigma_z / 2)  # RZ(π/4)

# RX(θ) gate: e^{-iθσx/2}
RX = expm(-1j * theta * sigma_x / 2)

# RY(θ) gate
sigma_y = np.array([[0, -1j], [1j, 0]])
RY = expm(-1j * theta * sigma_y / 2)

# Gate composition: verify e^A e^B ≠ e^{A+B} (non-commuting)
exp_sum = expm(-(1j) * (sigma_x + sigma_z))
exp_product = expm(-1j * sigma_x) @ expm(-1j * sigma_z)

# Trotter-Suzuki approximation for e^{-i(A+B)t}
n_steps = 10
dt = t / n_steps
trotter_approx = np.eye(2)
for _ in range(n_steps):
    trotter_approx = expm(-1j * sigma_z * dt / 2) @ expm(-1j * sigma_x * dt / 2) @ trotter_approx

# State evolution: |ψ(t)⟩ = U(t)|ψ(0)⟩
psi_0 = np.array([1, 0])  # |0⟩
psi_t = U_t @ psi_0  # evolve by time t

# Verify unitarity: U†U = I
unitarity = U_t.conj().T @ U_t

# Exponential of sin/cos
cos_sigma_z = np.cos(sigma_z)  # cos(A) = (e^{iA} + e^{-iA})/2 (used in expm)
sin_sigma_z = np.sin(sigma_z)  # sin(A) = (e^{iA} - e^{-iA})/(2i)

# Commutator: [A,B] = AB - BA
commutator_xz = sigma_x @ sigma_z - sigma_z @ sigma_x
# For commuting observables: e^{A+B} = e^A @ e^B


## Summary: The Schrödinger Equation is Matrix Exponentiation

✓ **e^A = Σₙ A^n/n!** always converges and is invertible  
✓ **Time evolution**: |ψ(t)⟩ = e^{-iHt/ℏ}|ψ(0)⟩ is the universal law of quantum dynamics  
✓ **Spectral method**: e^{-iΛt} computed on diagonal; inverse transforms back  
✓ **Rotation gates**: RX(θ), RY(θ), RZ(θ) are rotations generated by Pauli exponentials  
✓ **Trotter-Suzuki**: Approximates e^{-i(A+B)t} as product of simpler exponentials  
✓ **Commuting observables**: [A,B] = 0 ⟹ e^{A+B} = e^A e^B (simplification!)  
✓ **Variational ansatz**: Parametric quantum circuits are products of matrix exponentials  

Matrix exponentials connect differential equations to quantum gates—the foundation of quantum simulation and variational algorithms.